# 🎺 Parts Activity Visualization

**Difficulty:** 🟡 Medium  
**Prerequisites:** [Piano Rolls](01_piano_rolls.ipynb), [Loading Scores](../01_beginner/04_loading_scores.ipynb)  
**Time:** 25-30 minutes  

---

## What You'll Learn

- How to visualize which instruments/parts play when
- How to use `plotPartsActivity()` for orchestration analysis
- How to track instrument density over time
- How to identify tutti vs. solo passages
- How to analyze orchestration techniques
- How to compare instrumentation across pieces

## Why This Matters

Orchestration is the art of **who plays what, when**. Understanding parts activity helps you:

- **Analyze orchestration decisions**: When does the composer use full orchestra vs. chamber textures?
- **Study instrumental combinations**: Which instruments play together?
- **Identify structural boundaries**: Changes in orchestration often mark formal sections
- **Learn from masters**: How did Beethoven, Mozart, Mahler use their orchestras?
- **Improve your own orchestration**: See what works and why

This is essential for:
- Composers learning orchestration
- Conductors understanding score structure
- Music analysts studying form and texture
- Researchers comparing orchestration practices

---

## Step 1: Setup and Import

In [ ]:
# Import maialib and visualization libraries
import maialib as ml
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

print(f"✅ Maialib {ml.__version__} loaded successfully!")
print("🎺 Ready to analyze orchestration!")

---

## Step 2: Basic Parts Activity Visualization

Let's start by visualizing which instruments play in each measure.

In [ ]:
# Load an orchestral score
beethoven = ml.Score('Beethoven_Symphony_5_mov_1.xml')

print("Score Information:")
print(f"  Title: {beethoven.getWorkTitle()}")
print(f"  Composer: {beethoven.getComposer()}")
print(f"  Number of parts: {beethoven.getNumParts()}")
print(f"  Measures: {beethoven.getNumMeasures()}")

# List all parts
print("\nInstruments in the score:")
for i in range(beethoven.getNumParts()):
    part = beethoven.getPart(i)
    print(f"  {i+1}. {part.getPartName()}")

# Create parts activity visualization
ml.plotPartsActivity(beethoven)

print("\n📊 Parts activity plot created!")

**What You're Seeing:**

- **X-axis**: Time (measure numbers)
- **Y-axis**: Instrument/part names
- **Colored blocks**: When each instrument is playing
- **Empty spaces**: When instruments are silent (rests)

**Musical Insight - Beethoven's Opening:**
- Notice the **famous unison opening** - all parts play together (vertical alignment)
- Then **strings dominate** while winds have rests
- **Antiphonal passages** - different instrument groups alternate
- **Build-up to tutti** - more instruments join for climax moments

**Orchestration Term**: **Tutti** = all instruments playing together. You can spot these as dense vertical sections!

---

## Step 3: Calculating Instrument Density

Let's quantify how many instruments play at each moment.

In [ ]:
def calculate_instrument_density(score):
    """
    Calculate how many parts are active in each measure.
    
    Returns:
        DataFrame with measure number and active parts count
    """
    
    num_measures = score.getNumMeasures()
    num_parts = score.getNumParts()
    
    density_data = []
    
    for measure_idx in range(num_measures):
        active_parts = 0
        
        for part_idx in range(num_parts):
            part = score.getPart(part_idx)
            
            if measure_idx < part.getNumMeasures():
                measure = part.getMeasure(measure_idx)
                notes = measure.getNotes()
                
                # Part is active if it has notes
                if len(notes) > 0:
                    active_parts += 1
        
        density_data.append({
            'measure': measure_idx + 1,
            'active_parts': active_parts,
            'density_pct': (active_parts / num_parts) * 100
        })
    
    return pd.DataFrame(density_data)

# Calculate density for Beethoven
density = calculate_instrument_density(beethoven)

print("Orchestration Density Analysis:")
print(f"\nFirst 20 measures:")
print(density.head(20).to_string(index=False))

# Statistics
print(f"\n📊 Overall Statistics:")
print(f"  Average active parts: {density['active_parts'].mean():.1f}")
print(f"  Maximum density: {density['active_parts'].max()} parts (measure {density.loc[density['active_parts'].idxmax(), 'measure']})")
print(f"  Minimum density: {density['active_parts'].min()} parts")
print(f"  Tutti passages (100% density): {(density['density_pct'] == 100).sum()} measures")

**Interpreting Density Data:**

**High Density (Many Parts Active)**:
- **Musical effect**: Power, richness, climax, emphasis
- **Common in**: Opening statements, climaxes, final chords
- **Risks**: Can become muddy if overused

**Low Density (Few Parts Active)**:
- **Musical effect**: Clarity, intimacy, space, transparency
- **Common in**: Transitions, solos, chamber-like passages
- **Benefits**: Creates contrast, gives listeners a break

**Variable Density**:
- **Creates interest** through constant change
- **Marks structure** - different sections have different densities
- **Prevents fatigue** - ear needs variety

---

## Step 4: Visualizing Density Over Time

Let's create a graph showing orchestral density evolution.

In [ ]:
# Calculate density
density = calculate_instrument_density(beethoven)

# Create visualization
fig = go.Figure()

# Add area plot for density
fig.add_trace(go.Scatter(
    x=density['measure'],
    y=density['active_parts'],
    fill='tozeroy',
    name='Active Parts',
    line=dict(color='steelblue', width=2),
    fillcolor='rgba(70, 130, 180, 0.3)'
))

# Add a line for average
avg_density = density['active_parts'].mean()
fig.add_hline(
    y=avg_density, 
    line_dash="dash", 
    line_color="red",
    annotation_text=f"Average: {avg_density:.1f} parts",
    annotation_position="right"
)

# Update layout
fig.update_layout(
    title=f"Orchestration Density: {beethoven.getWorkTitle()}",
    xaxis_title="Measure",
    yaxis_title="Number of Active Parts",
    height=500,
    hovermode='x unified'
)

fig.show()

print("📊 Density evolution graph created!")
print("\nHover over the graph to see exact values.")
print("Notice how density changes mark structural sections!")

**Reading the Density Graph:**

Look for these patterns:

1. **Sudden drops to zero**: Rests, grand pauses, dramatic silences
2. **Gradual build-ups**: Crescendos, adding instruments one by one
3. **Plateaus**: Extended passages with consistent orchestration
4. **Peaks**: Climax moments, tutti passages
5. **Regular oscillations**: Antiphonal writing (back-and-forth between groups)

**In Sonata Form** (like Beethoven Symphony No. 5, Mvt. 1):
- **Exposition**: Variable density, introducing themes
- **Development**: Often MORE variable, dramatic contrasts
- **Recapitulation**: Similar to exposition
- **Coda**: Often builds to maximum density

---

## Step 5: Analyzing Instrument Group Usage

Let's group instruments by family (strings, woodwinds, brass, percussion) and analyze their usage.

In [ ]:
def categorize_instruments(score):
    """
    Categorize each part into instrument families.
    """
    
    categories = {
        'strings': ['violin', 'viola', 'cello', 'bass', 'contrabass'],
        'woodwinds': ['flute', 'oboe', 'clarinet', 'bassoon', 'piccolo'],
        'brass': ['horn', 'trumpet', 'trombone', 'tuba', 'cornet'],
        'percussion': ['timpani', 'drum', 'cymbal', 'triangle', 'percussion'],
        'keyboard': ['piano', 'harpsichord', 'organ', 'celesta'],
        'voices': ['soprano', 'alto', 'tenor', 'bass', 'choir', 'vocal']
    }
    
    part_categories = []
    
    for i in range(score.getNumParts()):
        part = score.getPart(i)
        part_name = part.getPartName().lower()
        
        category = 'other'
        for cat, keywords in categories.items():
            if any(keyword in part_name for keyword in keywords):
                category = cat
                break
        
        part_categories.append({
            'index': i,
            'name': part.getPartName(),
            'category': category
        })
    
    return pd.DataFrame(part_categories)

# Categorize Beethoven's instruments
categories = categorize_instruments(beethoven)

print("Instrument Families in Beethoven Symphony No. 5:")
print(categories.to_string(index=False))

# Count by category
print("\n📊 Parts by Family:")
family_counts = categories['category'].value_counts()
for family, count in family_counts.items():
    print(f"  {family.capitalize()}: {count}")

# Create pie chart
fig = px.pie(
    values=family_counts.values,
    names=family_counts.index,
    title="Orchestration by Instrument Family"
)

fig.show()

**Classical Orchestra Instrumentation:**

**Beethoven's Early 19th Century Orchestra**:
- **Strings**: Foundation, 60-70% of parts (Violin I, II, Viola, Cello, Bass)
- **Woodwinds**: Color and melody, ~20% (Flutes, Oboes, Clarinets, Bassoons in pairs)
- **Brass**: Power and fanfares, ~10% (Horns, Trumpets, limited Trombones)
- **Percussion**: Rare, mainly Timpani

**Evolution Over Time**:
- **Classical** (Mozart, Haydn): Small, balanced, ~30 players
- **Early Romantic** (Beethoven): Expanded brass, ~50 players
- **Romantic** (Brahms, Tchaikovsky): Triple winds, ~80 players
- **Late Romantic** (Mahler, Strauss): Huge, 100+ players, exotic instruments

---

## Step 6: Tracking Instrument Family Activity

Let's see when each instrument family is most active.

In [ ]:
def analyze_family_activity(score, categories_df):
    """
    Calculate activity percentage for each instrument family by measure.
    """
    
    num_measures = score.getNumMeasures()
    families = categories_df['category'].unique()
    
    activity_data = []
    
    for measure_idx in range(num_measures):
        measure_activity = {'measure': measure_idx + 1}
        
        # Initialize all families to 0
        for family in families:
            measure_activity[family] = 0
        
        # Count active parts in each family
        for _, row in categories_df.iterrows():
            part_idx = row['index']
            family = row['category']
            
            part = score.getPart(part_idx)
            if measure_idx < part.getNumMeasures():
                measure = part.getMeasure(measure_idx)
                notes = measure.getNotes()
                
                if len(notes) > 0:
                    measure_activity[family] += 1
        
        activity_data.append(measure_activity)
    
    return pd.DataFrame(activity_data)

# Analyze family activity
family_activity = analyze_family_activity(beethoven, categories)

print("Instrument Family Activity:")
print(family_activity.head(20))

# Create stacked area chart
fig = go.Figure()

families = [col for col in family_activity.columns if col != 'measure']
colors = ['steelblue', 'orange', 'green', 'red', 'purple', 'brown']

for i, family in enumerate(families):
    if family in family_activity.columns:
        fig.add_trace(go.Scatter(
            x=family_activity['measure'],
            y=family_activity[family],
            name=family.capitalize(),
            stackgroup='one',
            fillcolor=colors[i % len(colors)]
        ))

fig.update_layout(
    title="Instrument Family Activity Over Time",
    xaxis_title="Measure",
    yaxis_title="Active Parts",
    height=600,
    hovermode='x unified'
)

fig.show()

print("\n📊 Stacked area chart created!")
print("Each color represents an instrument family.")
print("Height shows how many parts from that family are active.")

**Reading the Family Activity Chart:**

**String Dominance**:
- Strings form the base layer (always present in Classical music)
- Can play alone for chamber-like passages
- Foundation for tutti sections

**Woodwind Additions**:
- Add color and brightness
- Often carry melody while strings accompany
- Can play alone for lighter textures

**Brass Entrances**:
- Dramatic moments, fanfares, climaxes
- Not continuously present (would be too heavy)
- Strategic deployment for maximum impact

**Orchestration Techniques You'll See**:
1. **Stratification**: Different families active at different times
2. **Doubling**: Same melody in multiple families (adds power)
3. **Call and response**: Families alternate
4. **Blend**: All families together (tutti)

---

## Step 7: Comparing Orchestration Across Composers

Let's compare how different composers use their orchestras.

In [ ]:
# Load multiple orchestral scores
scores = {
    'Mozart': ml.Score('Mozart_Requiem_Introitus.mxl'),
    'Beethoven': ml.Score('Beethoven_Symphony_5_mov_1.xml'),
    'Dvorak': ml.Score('Dvorak_Symphony_9_mov_4.mxl'),
    'Mahler': ml.Score('Mahler_Symphony_1_mov_1.mxl'),
}

# Analyze orchestration characteristics
comparison_data = []

for name, score in scores.items():
    density = calculate_instrument_density(score)
    
    comparison_data.append({
        'Composer': name,
        'Total Parts': score.getNumParts(),
        'Avg Active Parts': density['active_parts'].mean(),
        'Max Density': density['active_parts'].max(),
        'Tutti %': (density['density_pct'] == 100).sum() / len(density) * 100
    })

# Create comparison DataFrame
df_comparison = pd.DataFrame(comparison_data)

print("Orchestration Comparison Across Composers:")
print(df_comparison.to_string(index=False))

# Visualize
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Orchestra Size', 'Average Density', 'Tutti Usage %')
)

# Total parts
fig.add_trace(
    go.Bar(x=df_comparison['Composer'], y=df_comparison['Total Parts'],
           marker_color='steelblue', name='Total Parts'),
    row=1, col=1
)

# Average density
fig.add_trace(
    go.Bar(x=df_comparison['Composer'], y=df_comparison['Avg Active Parts'],
           marker_color='orange', name='Avg Active'),
    row=1, col=2
)

# Tutti percentage
fig.add_trace(
    go.Bar(x=df_comparison['Composer'], y=df_comparison['Tutti %'],
           marker_color='crimson', name='Tutti %'),
    row=1, col=3
)

fig.update_layout(
    height=400,
    title_text="Comparative Orchestration Analysis",
    showlegend=False
)

fig.show()

**Expected Orchestration Trends:**

**Mozart (Late Classical, 1791)**:
- Moderate orchestra size (~20-30 parts with chorus)
- Balanced, efficient orchestration
- Moderate tutti usage (saves for climaxes)

**Beethoven (Early Romantic, 1808)**:
- Larger orchestra (~40 parts)
- More dramatic contrasts (solo vs. tutti)
- Strategic tutti for maximum impact

**Dvořák (Late Romantic, 1893)**:
- Large orchestra (~50-60 parts)
- Rich, colorful orchestration
- More continuous use of full orchestra

**Mahler (Late Romantic, 1888-1896)**:
- **Huge** orchestra (80-100+ parts)
- Chamber-like passages despite large forces
- Lower average density (uses size for COLOR not volume)
- Less tutti % (saves massive sound for special moments)

**The Mahler Paradox**: Mahler wrote for HUGE orchestras but often used fewer instruments simultaneously than Classical composers! He wanted timbral variety, not just loudness.

---

## 🎯 Practice Exercises

### Exercise 1: Identify Tutti vs. Chamber Passages

Load any orchestral score and find:
1. The measure with maximum orchestral density (biggest tutti)
2. The measure with minimum density (most chamber-like)
3. Calculate what % of the piece uses full orchestra

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
score = ml.Score('Strauss_Blue_Danube_Waltz.mxl')

# Calculate density
density = calculate_instrument_density(score)

# Find extremes
max_row = density.loc[density['active_parts'].idxmax()]
min_row = density.loc[density['active_parts'].idxmin()]

print(f"Maximum tutti: {max_row['active_parts']} parts in measure {max_row['measure']}")
print(f"Minimum density: {min_row['active_parts']} parts in measure {min_row['measure']}")

# Calculate full orchestra usage
full_orch_pct = (density['density_pct'] == 100).sum() / len(density) * 100
print(f"Full orchestra used: {full_orch_pct:.1f}% of the time")

# Visualize
ml.plotPartsActivity(score)
```
</details>

---

### Exercise 2: Track a Single Instrument

Choose one instrument (e.g., first violin) and calculate:
- How many measures it plays
- How many measures it rests
- Its "activity percentage"

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
score = ml.Score('Beethoven_Symphony_5_mov_1.xml')

# Get first violin part
first_violin = score.getPart(0)
part_name = first_violin.getPartName()

# Count active measures
active_measures = 0
rest_measures = 0

for i in range(first_violin.getNumMeasures()):
    measure = first_violin.getMeasure(i)
    notes = measure.getNotes()
    
    if len(notes) > 0:
        active_measures += 1
    else:
        rest_measures += 1

total = active_measures + rest_measures
activity_pct = (active_measures / total) * 100

print(f"Instrument: {part_name}")
print(f"Active measures: {active_measures}")
print(f"Rest measures: {rest_measures}")
print(f"Activity: {activity_pct:.1f}%")
```
</details>

---

### Exercise 3: Compare String vs. Wind Usage

Calculate what percentage of the time strings are active vs. winds (woodwinds + brass).

In [ ]:
# YOUR CODE HERE


<details>
<summary>Click to see solution</summary>

```python
score = ml.Score('Mozart_Requiem_Introitus.mxl')

# Categorize instruments
categories = categorize_instruments(score)

# Get family activity
family_activity = analyze_family_activity(score, categories)

# Calculate percentages
string_active = (family_activity['strings'] > 0).sum()
woodwind_active = (family_activity['woodwinds'] > 0).sum()
brass_active = (family_activity['brass'] > 0).sum()

total_measures = len(family_activity)
winds_active = ((family_activity['woodwinds'] > 0) | (family_activity['brass'] > 0)).sum()

print(f"Strings active: {string_active}/{total_measures} measures ({string_active/total_measures*100:.1f}%)")
print(f"Winds active: {winds_active}/{total_measures} measures ({winds_active/total_measures*100:.1f}%)")
print(f"String dominance: {string_active > winds_active}")
```
</details>

---

## ✅ Summary

Congratulations! You can now analyze and visualize orchestration. You've learned:

- ✅ How to use `plotPartsActivity()` to see when instruments play
- ✅ How to calculate instrument density over time
- ✅ How to categorize instruments by family
- ✅ How to track instrument family activity
- ✅ How to identify tutti vs. chamber passages
- ✅ How to compare orchestration across composers and eras

## Key Concepts

- **Parts activity** = which instruments play when
- **Orchestral density** = how many instruments play simultaneously
- **Tutti** = all instruments together (maximum power)
- **Chamber texture** = few instruments (clarity and intimacy)
- **Instrument families**: strings, woodwinds, brass, percussion
- **Orchestration evolved** from small Classical to massive Late Romantic orchestras

## 🎯 Next Steps

Apply your orchestration knowledge:

1. **[Combining Visualizations](06_combining_visualizations.ipynb)** - Create comprehensive analyses
2. **[Form and Structure Analysis](../05_analysis/05_form_and_structure_analysis.ipynb)** - Use orchestration to identify sections
3. **[Creating Multi-Part Scores](../02_intermediate/06_multi_part_scores.ipynb)** - Apply what you've learned

---

<div align="center">

**See the orchestra breathe! 🎺**

*Parts activity reveals the ebb and flow of instrumental forces - the composer's paintbrush across the sonic canvas*

</div>